# Download Commit Comments with Kaiaulu (Notebook 4)

This notebook shows how to download GitHub commit comments using Kaiaulu’s `download_github_events.Rmd` notebook in the `/vignettes` folder.

### Planned Output

1. A parsed commit-comments CSV saved to `rawdata/github/{owner}/{repo}/{owner}_{repo}_commit_comments.csv` in Kaiaulu

### Step 1: Confirm your working directory

1. Open the Kaiaulu project in RStudio.
2. Run `getwd()` in the R console to check your current working directory.
3. If the directory is not Kaiaulu, set it with `setwd()` (for example, `setwd("~/Desktop/github/kaiaulu")`).

### Step 2: Create a personal access token

This workflow makes many GitHub API requests, so you need a personal access token.

Follow the [GitHub documentation](https://docs.github.com/en/free-pro-team@latest/github/authenticating-to-github/creating-a-personal-access-token#:~:text=Creating%20a%20token.%201%20Verify%20your%20email%20address%2C,able%20to%20see%20the%20token%20again.%20More%20items) and create a **classic** token:

1. Go to **GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)**.
2. Select **Generate new token (classic)**.
3. Add a note (for example, "Download GitHub commit + PR comments via Kaiaulu").
4. Enable the `public_repo` scope for public repositories.
5. Generate the token, then copy and store it securely.

Save the token in `~/.ssh/github_token` on your local machine.

### Step 3: Run `download_github_events.Rmd` chunks in RStudio

Run the following chunks in **RStudio**. These chunks should already exist in `download_github_events.Rmd`.

### Chunk 1: Set up dependencies

---
```{r warning=FALSE,message=FALSE}
rm(list = ls())
require(kaiaulu)
require(data.table)
require(jsonlite)
require(knitr)
```
---

### Chunk 2: Set required parameters

Replace `kaiaulu.yml` with the `.yml` file for the project you want to process. You created these files in Step 4 of `03_scale_config_files.ipynb`.

---
```{r}
conf <- parse_config("../conf/kaiaulu.yml")
owner <- get_github_owner(conf, "project_key_1") # Has to match github organization (e.g. github.com/sailuh)
repo <- get_github_repo(conf, "project_key_1") # Has to match github repository (e.g. github.com/sailuh/perceive)
save_path_issue_or_pr_comments <- path.expand(get_github_issue_or_pr_comment_path(conf, "project_key_1"))
save_path_issue_event <- get_github_issue_event_path(conf, "project_key_1")
save_path_commit <- get_github_commit_path(conf, "project_key_1")
save_path_commit_comments <- get_github_commit_comment_path(conf, "project_key_1")

# your file github_token contains the GitHub token API obtained in the steps above
token <- scan("~/.ssh/github_token",what="character",quiet=TRUE)
```
---

### Chunk 3: Download Commit Comments

This downloads commit-comment JSON files into `rawdata` in your current working directory. The runtime depends on how many comments the project has.

**IMPORTANT:** This chunk uses `gh_next()` to fetch paginated results and expects `gh` version 1.2.0. If you see a `gh_next()` paging bug (for example, repeated writes to the same page), downgrade to `gh` 1.2.0.

---

```{r Collect all project commit comments, eval = FALSE}
dir.create(save_path_commit_comments, recursive = TRUE, showWarnings = FALSE)
gh_response <- github_api_project_commit_comments(owner,repo,token)
github_api_iterate_pages(token,gh_response,save_path_commit_comments,prefix="commit_comments")
```

---

### Chunk 4: Parse Commit Comments

After all JSON files are downloaded, run the **Parsing Raw Data to Csv** chunk for commit comments. You should see a table named `all_commit_comments` in your R environment with columns such as `comment_id`, `commit_id`, `author_login`, `author_id`, `line`, `created_at`, and `updated_at`.

---

```{r}
all_commit_comments <- lapply(list.files(save_path_commit_comments,full.names = TRUE),read_json)
all_commit_comments <- lapply(all_commit_comments,github_parse_project_commit_comments)
all_commit_comments <- rbindlist(all_commit_comments,fill=TRUE)

kable(head(all_commit_comments))

# Save the data table for commit comments as a CSV
out_csv <- file.path(dirname(save_path_commit_comments), paste0(owner, "_", repo, "_commit_comments.csv"))
data.table::fwrite(all_commit_comments, out_csv)
cat("Saved:", out_csv, "\n")
```

---

### Final Output

Final output path:
`rawdata/github/{owner}/{repo}/{owner}_{repo}_commit_comments.csv`

### When to move on to Notebook 5

Move to Notebook 5 after all of the following are true:

1. The commit-comment JSON files have been downloaded successfully.
2. The parsed table `all_commit_comments` looks correct in RStudio.
3. The CSV file exists at:
   `rawdata/github/{owner}/{repo}/{owner}_{repo}_commit_comments.csv`
4. Spot-check a few rows to confirm key fields (such as `comment_id`, `commit_id`, and `author_login`) are populated as expected.